# 10 Quoting and expansion

<div class="bp-banner">
  <div class="bp-series">Introduction to the Bash Shell</div>
  <div style="display:flex;align-items:baseline;gap:14px;flex-wrap:wrap;">
    <span class="bp-title">Part III — From commands to scripts</span>
    <span class="bp-meta">Notebook&nbsp;10</span>
  </div>
  <div style="margin-top:10px;max-width:62ch;color:#46506b;">
    How the shell rewrites your command line before it runs, and how quoting
    lets you control that rewriting. The most common source of subtle bugs.
  </div>
  <div class="bp-rule" style="display:flex;justify-content:space-between;flex-wrap:wrap;gap:8px;">
    <span class="bp-meta">Raymond Amador</span>
    <span class="bp-meta">v1.0.0&nbsp;·&nbsp;CC&nbsp;BY&nbsp;4.0 (text) / MIT (code)</span>
  </div>
</div>

In [1]:
# Hidden setup: stand at the repo root, source the validation gate. data/ is
# read-only; anything we create (including a file with a space in its name) goes
# into a fresh scratch/.
ROOT="$PWD"; while [ ! -f "$ROOT/tools/check.sh" ] && [ "$ROOT" != "/" ]; do ROOT="$(dirname "$ROOT")"; done
source "$ROOT/tools/check.sh"
set +H
cd "$ROOT"

## What this notebook is about

You have met this idea three times already and never named it. In Notebook 5 the
shell expanded `*.xyz` before `ls` ran. In Notebooks 6 and 8 you single-quoted
your `grep` and `awk` programs so the shell would leave the `$` alone. Both are
the same fact: **before any command runs, the shell rewrites your command line.**

This notebook names that fact, lays out the full rewriting, and, most usefully,
shows how **quoting** lets you control it. Getting this wrong is the single most
common subtle bug in the shell, so it earns a whole notebook here at the start of
Part III, right before you begin writing scripts of your own. (As ever: this is all
filenames and values, **no physics.**)

## A. A bug, to start

Make a file whose name contains a space, and put its name in a variable:

In [2]:
cd "$ROOT"; rm -rf scratch; mkdir -p scratch
printf 'frame data\n' > "scratch/my run.xyz"

In [3]:
f="scratch/my run.xyz"

Now look closely. We pass `$f`, unquoted, to a command, and print each argument
the command actually receives:

In [4]:
printf 'argument: [%s]\n' $f

argument: [scratch/my]


argument: [run.xyz]


The command received **two** arguments, not one. The space in the filename tore
`$f` into `scratch/my` and `run.xyz`. Try to use it and it breaks: `cat` goes
looking for two files that do not exist:

In [5]:
cat $f 2>/dev/null || echo '(failed — the unquoted variable split into two names)'

(failed — the unquoted variable split into two names)


Hold that bug in mind. By the end of the notebook it will be a one-character fix
you never forget.

## B. The model: the shell rewrites your line

Here is what really happens. When you press Enter, the shell does **not** hand your
line to the command as you typed it. It first rewrites the line through a fixed
sequence of **expansions**, each one substituting something in, and only then runs
the command on the *result*. You have already met most of them:

<div class="bp-card">
  <span class="bp-card-cmd">The expansions, in order</span> — <span class="bp-card-job">the shell applies these to your line before the command runs; the command sees only the result.</span>
  <table>
    <tr><td>{a,b}  {1..9}</td><td>brace expansion: generate strings (Notebook 5)</td></tr>
    <tr><td>~</td><td>tilde: your home directory (Notebook 2)</td></tr>
    <tr><td>$var  ${var}</td><td>parameter expansion: the value of a variable</td></tr>
    <tr><td>$(cmd)</td><td>command substitution: the output of a command</td></tr>
    <tr><td>$((expr))</td><td>arithmetic expansion: integer arithmetic</td></tr>
    <tr><td><b>word splitting</b></td><td>split the <i>unquoted</i> results on whitespace: <b>the culprit in §A</b></td></tr>
    <tr><td>*  ?  [ ]</td><td>pathname (glob) expansion: match filenames (Notebook 5)</td></tr>
  </table>
</div>

The whole notebook hangs off the bold row. Word splitting is what chopped `$f` in
two: it splits *unquoted* expansion results on spaces. Which means the fix is to
stop the result from being unquoted. That is what quoting is for.

## C. Quoting — the control

Three tools, three behaviours:

<div class="bp-card">
  <span class="bp-card-cmd">Quoting</span> — <span class="bp-card-job">shell syntax (not a command) that controls which expansions happen.</span>
  <table>
    <tr><td>'single'</td><td>literal: <b>no expansion at all</b>, the text is taken exactly as written</td></tr>
    <tr><td>"double"</td><td>allow <code>$</code>-expansions, but <b>suppress word splitting and globbing</b></td></tr>
    <tr><td>\\  (backslash)</td><td>escape: protect the single next character</td></tr>
  </table>
</div>

Single quotes turn *everything* off: the shell does not touch what is inside:

In [6]:
echo '$HOME and *.xyz are left exactly as typed'

$HOME and *.xyz are left exactly as typed


That is precisely **why you single-quote `grep` and `awk` programs** (Notebooks 6
and 8): a `$1` inside single quotes reaches `awk` untouched. Double quotes are the
middle ground: variables *do* expand, but the result is kept as one piece:

In [7]:
echo "your home is $HOME"

your home is /home/runner


And a backslash protects just the next character: here, a literal dollar sign:

In [8]:
echo "a price tag: \$5"

a price tag: $5


## D. The one habit, and the traps

Everything above points at a single rule, and it is the highest-value habit in all
of shell scripting:

:::{admonition} ⚠ Always double-quote your variable expansions
:class: danger
Write `"$f"`, not `$f`. An unquoted variable that happens to contain a **space**
becomes two arguments (the §A bug); one that contains a **`*`** gets glob-expanded
against the current directory, matching who-knows-what. Quietly, with no error,
your command operates on the wrong thing, and when that command is `rm`, the
result is the kind of story people tell years later. The fix costs two characters.
Quote every `"$var"` and every `"$@"`, every time, and a whole category of bugs
simply never happens.
:::

Watch the fix retire the §A bug: the *same* command, the *only* difference being
the quotes:

In [9]:
printf 'argument: [%s]\n' "$f"

argument: [scratch/my run.xyz]


In [10]:
cat "$f"

frame data


One argument, the right file. A few more traps worth knowing while we are here:

- **Empty or unset variables** vanish when unquoted, which can turn `command $x`
  into just `command`. A default value guards against it: `${x:-fallback}` (more in
  §E).
- **`"$@"`** is the right way to forward "all the arguments, each kept whole";
  unquoted `$@` and `$*` both word-split and will mangle any argument with a space.
  (You will use `"$@"` constantly once you write scripts in Notebook 12.)
- **Backticks** `` `cmd` `` are the old form of command substitution. Use **`$(cmd)`**
  instead: it nests cleanly and reads better.

## E. The workhorse `${ }` forms

Parameter expansion does more than fetch a value. A curated handful of `${…}` forms
covers almost everything you will reach for, most of it about reshaping filenames,
which is exactly what scripts spend their time doing.

<div class="bp-card">
  <span class="bp-card-cmd">Parameter expansion</span> — <span class="bp-card-job">the workhorse <code>${…}</code> forms. Case conversion, substrings, indirection, and arrays are out of scope.</span>
  <table>
    <tr><td>${var:-default}</td><td>use <i>default</i> if var is unset or empty</td></tr>
    <tr><td>${#var}</td><td>the length of var</td></tr>
    <tr><td>${var%.xyz}</td><td>remove a matching <b>suffix</b> (strip an extension)</td></tr>
    <tr><td>${var#prefix}</td><td>remove a matching <b>prefix</b></td></tr>
    <tr><td>${var/old/new}</td><td>substitute the first <i>old</i> with <i>new</i></td></tr>
    <tr><td>$(cmd)</td><td>command substitution: capture a command's output</td></tr>
    <tr><td>$((expr))</td><td>arithmetic: <b>integer only</b></td></tr>
  </table>
</div>

Suffix-stripping is the daily one: take an input name, drop its extension, build the
matching output name. This is most of what a data script does between files:

In [11]:
name="lj38-relaxed.xyz"
echo "${name%.xyz}.dat"

lj38-relaxed.dat


Command substitution drops a command's *output* into your line, and arithmetic does
integer math:

In [12]:
count=$(ls data | wc -l)
echo "data/ holds $count entries"

data/ holds 6 entries


In [13]:
echo "17 divided by 5 is $(( 17 / 5 ))"

17 divided by 5 is 3


Note that `3`: `$(( ))` is **integer only**, and it *truncates* (it does not round).
The moment you need a real number (a mean, a ratio with decimals) the shell cannot
help, and you reach back for `awk` (Notebook 8) or `bc`:

In [14]:
grep 'Total FORCE_EVAL' data/logs/gr2hno3-nvt.log | grep -oE '\-[0-9]+\.[0-9]+' | awk '{ s += $1; n++ } END { printf "mean = %.4f\n", s/n }'

mean = -143.4444


That is the honest division of labour: the shell for integers and string-shaping,
`awk` for the floating-point.

## Exercises

The point of this set is to *feel* the bugs and fix them. Anything that creates a
file works in a fresh `scratch/`; the rest is read-only.

### Warm-up 1 (worked) — The three quote types

`echo` a variable with no quotes, single quotes, and double quotes, and watch
expansion switch off and on.

In [15]:
cd "$ROOT"

In [16]:
# (solution hidden on the public site)


hello


hello from /home/runner


$greeting stays literal


In [17]:
greeting="hello"
expanded="$(echo "$greeting")"
literal="$(echo '$greeting')"
check '[ "$expanded" = "hello" ] && [ "$literal" = "\$greeting" ]' \
      "double quotes expanded the variable; single quotes kept it literal"

✓ double quotes expanded the variable; single quotes kept it literal


### Warm-up 2 (your turn) — Command substitution

Capture the output of a command into a variable with `$(…)`, then use it in a
sentence. Count the entries in `data/` and report the number.

In [18]:
cd "$ROOT"

In [19]:
# (solution hidden on the public site)


data/ has 6 entries


In [20]:
n=$(ls data | wc -l)
check '[ -n "$n" ] && [ "$n" -eq 6 ]' \
      "the command's output was captured into the variable"

✓ the command's output was captured into the variable


### Applied 1 (your turn) — The word-splitting bug, then the fix

The centrepiece. In `scratch/`, make a file whose name has a space. Operate on it
through an **unquoted** variable and watch it fail; then quote the variable and
watch it work.

In [21]:
cd "$ROOT"; rm -rf scratch; mkdir -p scratch
printf 'content\n' > "scratch/a file.txt"

In [22]:
# (solution hidden on the public site)


unquoted: failed (split into two names)


content


In [23]:
f="scratch/a file.txt"
unquoted_ok=$(cat $f 2>/dev/null && echo yes); quoted_ok=$(cat "$f" >/dev/null 2>&1 && echo yes)
check '[ "$unquoted_ok" != "yes" ] && [ "$quoted_ok" = "yes" ]' \
      "the unquoted form failed and the quoted form succeeded"

✓ the unquoted form failed and the quoted form succeeded


### Applied 2 (your turn) — The workhorse `${ }` forms

Given a filename, build derived strings: strip the `.xyz` to make a `.dat` output
name (`${f%.xyz}`), get the name's length (`${#f}`), and supply a default for an
unset variable (`${u:-none}`).

In [24]:
cd "$ROOT"

In [25]:
# (solution hidden on the public site)


lj38-relaxed.dat


length is 16


label is none


In [26]:
f="lj38-relaxed.xyz"
check '[ "${f%.xyz}.dat" = "lj38-relaxed.dat" ] && [ "${#f}" -eq 16 ] && [ "${u:-none}" = "none" ]' \
      "the output name, length, and default were all derived correctly"

✓ the output name, length, and default were all derived correctly


### Applied 3 (worked) — Arithmetic and its limit

Integer arithmetic with `$(( ))` (note the truncation), then a *mean* (which needs
floats) with `awk`.

In [27]:
cd "$ROOT"

In [28]:
# (solution hidden on the public site)


integer: 66


float mean: -143.4444


In [29]:
mean="$(grep 'Total FORCE_EVAL' data/logs/gr2hno3-nvt.log | grep -oE '\-[0-9]+\.[0-9]+' | awk '{s+=$1;n++} END{printf "%.4f", s/n}')"
check '[ "$(( 2520 / 38 ))" -eq 66 ] && [ "$mean" = "-143.4444" ]' \
      "integer division truncated to 66, and awk gave the float mean -143.4444"

✓ integer division truncated to 66, and awk gave the float mean -143.4444


### Composite — putting it together (build commands safely)

Build output filenames from a set of trajectory files (**one with a space in its
name**) using trimming and properly quoted expansions. (The `for … in …; do … done`
loop here just means "for each file"; it gets its full introduction in Notebook 13.
The lesson now is that *quoting* makes the loop space-safe.)

In [30]:
cd "$ROOT"; rm -rf scratch; mkdir -p scratch
printf 'x\n' > scratch/run_1.xyz
printf 'x\n' > scratch/run_2.xyz
printf 'x\n' > "scratch/run 3.xyz"

In [31]:
# (solution hidden on the public site)


'run 3.dat'  'run 3.xyz'   run_1.dat   run_1.xyz   run_2.dat   run_2.xyz


In [32]:
check '[ -f scratch/run_1.dat ] && [ -f "scratch/run 3.dat" ] && [ "$(ls scratch/*.dat | wc -l)" -eq 3 ]' \
      "a .dat was built for every .xyz, including the one whose name has a space"

✓ a .dat was built for every .xyz, including the one whose name has a space


### Optional stretch (your turn) — An expansion-order puzzle

A glob stored in a *quoted* variable does not expand, because globbing happens to
the line, not to a quoted value. Confirm it: put `*.xyz` in a variable, then echo it
quoted (stays literal) versus letting it expand. Predict each before running.

In [33]:
cd "$ROOT"; rm -rf scratch; mkdir -p scratch; touch scratch/x.xyz scratch/y.xyz

In [34]:
# (solution hidden on the public site)


*.xyz


x.xyz y.xyz


In [35]:
cd "$ROOT/scratch"; p='*.xyz'
quoted="$(echo "$p")"
words="$(echo $p | wc -w)"
cd "$ROOT"
check '[ "$quoted" = "*.xyz" ] && [ "$words" -eq 2 ]' \
      "quoted, the pattern stayed literal; unquoted, it expanded to the two files"

✓ quoted, the pattern stayed literal; unquoted, it expanded to the two files


## Outlook

You now control how the shell reads what you type: the foundation that keeps a
script from breaking the first time a filename has a space in it. That foundation
matters because Part III is about turning your one-liners into **scripts you keep
and re-run**. The next missing piece is mechanical: what makes a file *runnable* in
the first place. Next (Notebook 11): permissions and execution.

<div class="bp-banner" style="margin-top:30px;">
  <div class="bp-series">No new commands this time</div>
  <div style="font-size:14.5px;line-height:1.55;max-width:66ch;">
    Quoting and expansion are shell <b>syntax</b>, not commands, so this notebook
    adds nothing to the Compendium: its reference is the three tables above. The
    next command card returns in Notebook 11.
  </div>
</div>

<div class="bp-banner" style="margin-top:18px;">
  <div class="bp-series">Take this notebook with you</div>
  <div style="font-size:14.5px;line-height:1.55;max-width:66ch;">
    Open a <b>live terminal</b> from the &ldquo;Practice here&rdquo; box in any
    section to run everything yourself; nothing to install. The published
    notebooks ship <b>without worked solutions</b>; if you would like the
    reference solutions (to teach from or to check your own work), get in
    touch: <a href="mailto:hello@ramador.me">hello@ramador.me</a>.
  </div>
</div>